In [21]:
import sys
!{sys.executable} -m pip install -U -q google-cloud-aiplatform pandas==2.2.2 seaborn moviepy sounddevice 

### Setup Vertex AI

In [2]:
from google import genai

# Authenticate user for Colab environment

PROJECT_ID = "demoproject-cbbe7" 
LOCATION = "us-central1"
# Vertex AI Embedding Model
EMBEDDING_MODEL_VERTEX_AI = "gemini-embedding-2-preview"

client = genai.Client(
    vertexai=True, project=PROJECT_ID, location=LOCATION
)


### Create audio database

In [ ]:
from pathlib import Path

songs_dir = Path("songs")
audio_extensions = {".mp3"}

song_database = [
    {
        "song_id": idx,
        "song_name": song_file.stem,
        "song_path": str(song_file),
        "audio_embeddings": None,
        "meta_data": {
            "file_extension": song_file.suffix.lower(),
            "file_name": song_file.name,
            "tags": ['song']
        },
    }
    for idx, song_file in enumerate(sorted(songs_dir.iterdir()), start=1)
    if song_file.is_file() and song_file.suffix.lower() in audio_extensions
]

song_database

### Creating Embedding of a song

In [ ]:
from pathlib import Path
import mimetypes
import time
from google.genai import types

from moviepy import AudioFileClip
import tempfile
import os
from pathlib import Path



def _extract_embedding_vector(result):
    if hasattr(result, "embeddings") and result.embeddings:
        first = result.embeddings[0]
        if hasattr(first, "values"):
            return list(first.values)
        if isinstance(first, dict) and "values" in first:
            return list(first["values"])
    raise ValueError("No embedding vector found in response.")


def _embed_audio_bytes_with_retry(audio_bytes, mime_type, max_retries=5, base_delay=3):
    last_error = None
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model=EMBEDDING_MODEL_VERTEX_AI,
                contents= types.Part.from_bytes(
                        data=audio_bytes,
                        mime_type=mime_type,
                    )
                
            )
            print("Successfully embedded audio bytes.")
            return _extract_embedding_vector(result)
        except Exception as exc:
            print("Failed to generate audio embeddings. \n", exc)
            last_error = exc
            err = str(exc).upper()
            is_retryable = "429" in err or "RESOURCE_EXHAUSTED" in err
            if (not is_retryable) or attempt == max_retries - 1:
                raise
            time.sleep(base_delay * (2 ** attempt))
    raise last_error

print("Starting to embed audio files from the song database...")

for song in song_database:
    audio_path = Path(song["song_path"])
    try:
        # 1. Load the audio file
        audio = AudioFileClip(str(audio_path))
        
        # 2. Slice the first 60 seconds (or less if the song is shorter)
        end_time = min(60, audio.duration)
        audio_chunk = audio.subclipped(0, end_time)
        
        # 3. Create a temporary file to hold the MP3 chunk
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        temp_file.close() # Close it so moviepy can safely write to it
        
        # 4. Export the chunk to the temp file (logger=None hides the progress bar spam)
        audio_chunk.write_audiofile(temp_file.name, logger=None)
        
        # 5. Read the bytes back into memory for Vertex AI
        with open(temp_file.name, "rb") as f:
            audio_bytes = f.read()
            
        # 6. Clean up the temp file
        os.unlink(temp_file.name)
        
        # 7. Make the API call
        mime_type = "audio/mp3"
        embedding_vector = _embed_audio_bytes_with_retry(audio_bytes, mime_type)

        song["audio_embeddings"] = embedding_vector
        song["meta_data"]["embedding_source"] = "audio_bytes"
        song["meta_data"].pop("embedding_error", None)
        print("Successfully embedded:", song["song_name"],"\n",len(song["audio_embeddings"]),"\n",song["audio_embeddings"],"\n\n")
    
    except Exception as exc:
        print("Failed to embed audio bytes.", exc)
        song["audio_embeddings"] = None
        song["meta_data"]["embedding_error"] = str(exc)




Starting to embed audio files from the song database...
Successfully embedded audio bytes.
Successfully embedded: Aditya Sharma - BAD IDEA  Electronic Pop  NCS - Copyright Free Music 
 [-0.01445398, 0.01753738, 0.011283197, -0.01014649, -0.009410275, 0.0016182686, 0.0002577307, 0.008655562, 0.0023490193, -0.06322659, -0.0028648104, 0.01112596, -0.0103927655, -0.0069172382, 0.0064598275, 2.4339477e-05, 0.0137403505, -0.015195785, 0.0049934234, -0.008127478, -0.0017690196, -0.003919227, 0.036249306, 0.015000436, -0.0033464008, -0.003513644, 0.0014302741, 0.0007477619, -0.0010584884, 0.13056296, -0.013263134, -0.007987088, 0.008224837, -0.023566402, 0.0046888734, -0.008727411, 0.019444829, 0.008402169, -0.0007396531, 0.01307431, -0.0076278686, -0.018778266, -0.02342658, -0.0086179925, -0.0024479544, 0.0032209852, -0.014507129, 0.019402185, -0.0030224188, 0.0053005316, 0.0073155602, -0.0036575038, -0.017433474, -0.00744805, -0.0073205875, 0.01760251, -0.010968959, 0.005940516, -0.010840868

In [16]:
{
    "total_songs": len(song_database),
    "with_audio_embeddings": sum(1 for s in song_database if s.get("audio_embeddings") is not None),
    "without_audio_embeddings": sum(1 for s in song_database if s.get("audio_embeddings") is None),
    "sample_embedding_dim": next((len(s["audio_embeddings"]) for s in song_database if s.get("audio_embeddings") is not None), None),
}

{'total_songs': 6,
 'with_audio_embeddings': 6,
 'without_audio_embeddings': 0,
 'sample_embedding_dim': 3072}

In [20]:
[
    {
        "song_id": s["song_id"],
        "song_name": s["song_name"],
        "has_embedding": s.get("audio_embeddings") is not None,
        "error": s.get("meta_data", {}).get("embedding_error"),
    }
    for s in song_database
]

[{'song_id': 1,
  'song_name': 'Aditya Sharma - BAD IDEA  Electronic Pop  NCS - Copyright Free Music',
  'has_embedding': True,
  'error': None},
 {'song_id': 2,
  'song_name': 'CERES, TAME - Pull Me Down  Techno  NCS - Copyright Free Music',
  'has_embedding': True,
  'error': None},
 {'song_id': 3,
  'song_name': 'LXNGVX, Warriyo - Mortals Funk Remix  NCS - Copyright Free Music',
  'has_embedding': True,
  'error': None},
 {'song_id': 4,
  'song_name': 'studiokolomna-no-copyright-music-483817',
  'has_embedding': True,
  'error': None},
 {'song_id': 5,
  'song_name': 'Syn Cole, Nakama - Feel Good Funk  Funk  NCS - Copyright Free Music',
  'has_embedding': True,
  'error': None},
 {'song_id': 6,
  'song_name': 'the_mountain-no-copyright-music-494934',
  'has_embedding': True,
  'error': None}]

### Taking User input

In [70]:
from pathlib import Path
from datetime import datetime
import wave
import sounddevice as sd

# Configure recording
duration_seconds = 10
sample_rate = 16000
channels = 1

all_devices = sd.query_devices()
input_devices = [
    {"global_index": i, "name": d["name"], "max_input_channels": d["max_input_channels"]}
    for i, d in enumerate(all_devices)
    if d.get("max_input_channels", 0) > 0
]

if not input_devices:
    raise RuntimeError("No input audio device found.")

print("Available input devices (global index):")
for d in input_devices:
    print(
        f"{d['global_index']}: {d['name']} "
        f"(max_input_channels={d['max_input_channels']})"
    )

# Use a GLOBAL device index from the printed list above.
# Example for your NVIDIA Broadcast WASAPI device: 30
selected_device_global_index = 1

if selected_device_global_index not in {d["global_index"] for d in input_devices}:
    raise ValueError(
        "Selected device index is not a valid input device. "
        "Choose one from the printed global index list."
    )

selected_device_name = all_devices[selected_device_global_index]["name"]

recordings_dir = Path("recordings")
recordings_dir.mkdir(parents=True, exist_ok=True)

output_path = recordings_dir / f"mic_recording.wav"

print(
    f"Recording {duration_seconds}s from index {selected_device_global_index}: "
    f"{selected_device_name}"
)
audio_data = sd.rec(
    int(duration_seconds * sample_rate),
    samplerate=sample_rate,
    channels=channels,
    dtype="int16",
    device=selected_device_global_index,
)
sd.wait()

with wave.open(str(output_path), "wb") as wf:
    wf.setnchannels(channels)
    wf.setsampwidth(2)
    wf.setframerate(sample_rate)
    wf.writeframes(audio_data.tobytes())

saved_mic_audio = {
    "audio_path": str(output_path),
    "duration_seconds": duration_seconds,
    "sample_rate": sample_rate,
    "channels": channels,
    "device_index": selected_device_global_index,
    "device": selected_device_name,
}

saved_mic_audio

Available input devices (global index):
0: Microsoft Sound Mapper - Input (max_input_channels=2)
1: Microphone (2- Fifine Microphon (max_input_channels=1)
2: Microphone (Iriun Webcam) (max_input_channels=2)
3: Microphone (NVIDIA Broadcast) (max_input_channels=2)
4: Headset Microphone (Oculus Virt (max_input_channels=1)
5: Headset (Portronics Harmony Min (max_input_channels=1)
12: Primary Sound Capture Driver (max_input_channels=2)
13: Microphone (2- Fifine Microphone) (max_input_channels=1)
14: Microphone (Iriun Webcam) (max_input_channels=2)
15: Microphone (NVIDIA Broadcast) (max_input_channels=2)
16: Headset Microphone (Oculus Virtual Audio Device) (max_input_channels=1)
17: Headset (Portronics Harmony Mini ) (max_input_channels=1)
29: Microphone (Iriun Webcam) (max_input_channels=2)
30: Microphone (NVIDIA Broadcast) (max_input_channels=2)
31: Microphone (2- Fifine Microphone) (max_input_channels=1)
32: Headset Microphone (Oculus Virtual Audio Device) (max_input_channels=1)
33: Heads

{'audio_path': 'recordings\\mic_recording.wav',
 'duration_seconds': 10,
 'sample_rate': 16000,
 'channels': 1,
 'device_index': 1,
 'device': 'Microphone (2- Fifine Microphon'}

### Convert user audio input to Embedding

In [71]:
from pathlib import Path
import mimetypes
from google.genai import types



def _extract_embedding_vector(result):
    if hasattr(result, "embeddings") and result.embeddings:
        first = result.embeddings[0]
        if hasattr(first, "values"):
            return list(first.values)
        if isinstance(first, dict) and "values" in first:
            return list(first["values"])
    if hasattr(result, "embedding") and result.embedding is not None:
        emb = result.embedding
        if hasattr(emb, "values"):
            return list(emb.values)
        if isinstance(emb, dict) and "values" in emb:
            return list(emb["values"])
    raise ValueError("No embedding vector found in response.")


def _extract_text_response(resp):
    if hasattr(resp, "text") and resp.text:
        return resp.text
    if hasattr(resp, "candidates") and resp.candidates:
        content = getattr(resp.candidates[0], "content", None)
        parts = getattr(content, "parts", []) if content else []
        texts = [p.text for p in parts if hasattr(p, "text") and p.text]
        if texts:
            return "\n".join(texts)
    raise ValueError("Could not extract text from audio analysis response.")


if "saved_mic_audio" not in globals() or "audio_path" not in saved_mic_audio:
    raise ValueError("saved_mic_audio not found. Run the microphone recording cell first.")

audio_path = Path(saved_mic_audio["audio_path"])
if not audio_path.exists():
    raise FileNotFoundError(f"Recorded audio file not found: {audio_path}")

with open(audio_path, "rb") as f:
    audio_bytes = f.read()

mime_type = mimetypes.guess_type(str(audio_path))[0] or "audio/wav"

try:
    # Try direct audio embedding first.
    embed_result = client.models.embed_content(
        model=EMBEDDING_MODEL_VERTEX_AI,
        contents=[types.Part.from_bytes(data=audio_bytes, mime_type=mime_type)],
    )
    embedding_vector = _extract_embedding_vector(embed_result)
    embedding_source = "direct_audio"

except Exception:


    embedding_vector = _extract_embedding_vector(embed_result)
    embedding_source = "audio_summary_text"


saved_mic_audio["audio_embeddings"] = embedding_vector
saved_mic_audio["embedding_meta"] = {
    "source": embedding_source,
    "embedding_dim": len(embedding_vector),
    "mime_type": mime_type,
}

saved_mic_audio

{'audio_path': 'recordings\\mic_recording.wav',
 'duration_seconds': 10,
 'sample_rate': 16000,
 'channels': 1,
 'device_index': 1,
 'device': 'Microphone (2- Fifine Microphon',
 'audio_embeddings': [-0.016480539,
  -0.0028931461,
  0.010707859,
  0.013884849,
  0.0098270215,
  -0.0072040204,
  -0.005613961,
  -0.011918249,
  -0.0021050526,
  -0.05900044,
  0.0071907225,
  0.0066429507,
  -0.009477087,
  0.011846281,
  0.010509398,
  -0.010795351,
  0.022597982,
  -0.0022725395,
  0.009264948,
  -0.015407849,
  0.01518685,
  -0.005819464,
  0.02126144,
  0.0011385754,
  -0.0035631417,
  0.0020012981,
  0.00015613718,
  0.0014209411,
  -0.038897034,
  0.1451792,
  -0.00040617207,
  -0.0015275144,
  -0.011846423,
  -0.01757538,
  0.013640226,
  -0.010018086,
  -0.006116203,
  0.007396453,
  -0.027613273,
  -0.0024260704,
  -0.00510458,
  -0.02229909,
  -0.0041361772,
  -0.012357624,
  0.015061483,
  -0.0066152844,
  -0.016302776,
  0.006501276,
  0.0072781094,
  -0.0044992846,
  0.015346

### Search audio in Database

In [72]:
import numpy as np

# Ensure query embedding is available under saved_mic_audio['audio_embeddings'].
if "audio_embeddings" not in saved_mic_audio or saved_mic_audio.get("audio_embeddings") is None:
    if saved_mic_audio.get("embeddings") is not None:
        saved_mic_audio["audio_embeddings"] = saved_mic_audio["embeddings"]
    else:
        raise ValueError(
            "saved_mic_audio['audio_embeddings'] not found. Run the conversion-to-embedding cell first."
        )

query_embedding = np.asarray(saved_mic_audio["audio_embeddings"], dtype=np.float32)
query_norm = np.linalg.norm(query_embedding)
if query_norm == 0:
    raise ValueError("Query embedding norm is zero, cannot compute cosine similarity.")

cosine_similarity_results = []
dot_similarity_results = []

for idx, song in enumerate(song_database):
    song_embedding = song.get("audio_embeddings")
    if song_embedding is None:
        continue

    song_embedding = np.asarray(song_embedding, dtype=np.float32)
    if song_embedding.shape != query_embedding.shape:
        continue

    dot_similarity = float(np.dot(query_embedding, song_embedding))
    song_norm = np.linalg.norm(song_embedding)
    if song_norm == 0:
        continue
    cosine_similarity = float(dot_similarity / (query_norm * song_norm))

    cosine_similarity_results.append(
        {
            "song_name": song.get("song_name"),
            "index_from_database": idx,
            "similarity_score": round(cosine_similarity, 6),
        }
    )

    dot_similarity_results.append(
        {
            "song_name": song.get("song_name"),
            "index_from_database": idx,
            "similarity_score": round(dot_similarity, 6),
        }
    )

cosine_similarity_results = sorted(
    cosine_similarity_results,
    key=lambda x: x["similarity_score"],
    reverse=True,
)

dot_similarity_results = sorted(
    dot_similarity_results,
    key=lambda x: x["similarity_score"],
    reverse=True,
)

print("Top dot product similarity result:")

print(dot_similarity_results[0])

print("\nTop cosine similarity result:")
print(cosine_similarity_results[0])






Top dot product similarity result:
{'song_name': 'Syn Cole, Nakama - Feel Good Funk  Funk  NCS - Copyright Free Music', 'index_from_database': 4, 'similarity_score': 0.793435}

Top cosine similarity result:
{'song_name': 'Syn Cole, Nakama - Feel Good Funk  Funk  NCS - Copyright Free Music', 'index_from_database': 4, 'similarity_score': 0.793435}
